In [31]:
import os
from pathlib import Path
from uuid import uuid4
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [32]:
# Load OPENAI_API_KEY from .env
load_dotenv()

True

## 1. Set up Folder Path for Vector Store  ##

In [33]:
project_root = Path.cwd()
print(f"Project root: {project_root}")

collection_name = "demo_collection"
persist_directory = project_root / "chroma_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")


Project root: d:\LangChain Fundamentals\Retrievers in LangChain v1
Collection name: demo_collection
Persist directory: d:\LangChain Fundamentals\Retrievers in LangChain v1\chroma_db


## 2. Text in Document Format ##
<p>Here we have converted the Text in a proper Document Class. If you have a PDF or CSV then use corresponding Textloader to do the same.</p>

In [34]:
# Create a list of documents with metadata
docs = [
    Document(page_content="Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.",
             metadata={"topic": "space"}),
    Document(page_content="The International Space Station orbits Earth at about 400 km altitude and travels at 28,000 km/h.",
             metadata={"topic": "space"}),
    Document(page_content="Spacecraft use gravitational slingshots around planets to gain speed without burning extra fuel.",
             metadata={"topic": "space"}),
    Document(page_content="NASA's Voyager 1 is the farthest human-made object, now over 23 billion km from the Sun.",
             metadata={"topic": "space"}),
    Document(page_content="Solar sails use radiation pressure from sunlight to slowly propel spacecraft without fuel.",
             metadata={"topic": "space"}),
    Document(page_content="DNA is a double-helix molecule that carries the genetic instructions for all living organisms.",
             metadata={"topic": "biology"}),
    Document(page_content="Photosynthesis allows plants to convert sunlight, water, and CO2 into glucose and oxygen.",
             metadata={"topic": "biology"}),
    Document(page_content="The Roman Empire at its peak covered over 5 million square kilometers across three continents.",
             metadata={"topic": "history"}),
    Document(page_content="The printing press, invented by Gutenberg around 1440, revolutionised the spread of knowledge.",
             metadata={"topic": "history"}),
    Document(page_content="The Amazon River discharges more freshwater into the ocean than any other river on Earth.",
             metadata={"topic": "geography"}),
    Document(page_content="The Sahara Desert spans about 9.2 million square kilometers across northern Africa.",
             metadata={"topic": "geography"}),
    Document(page_content="Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.",
             metadata={"topic": "biology"}),
    Document(page_content="Training a deep learning model involves iteratively adjusting weights using gradient descent to minimise the loss.", metadata={"topic": "deep learning"}),
    Document(page_content="Deep learning models are optimised through gradient descent, which updates weights in the direction that reduces the training loss.", metadata={"topic": "deep learning"}),
    Document(page_content="Gradient descent is the core optimisation technique in deep learning, guiding weight updates based on computed gradients of the loss.", metadata={"topic": "deep learning"}),
    Document(page_content="Dropout randomly disables a fraction of neurons during training to prevent overfitting in deep networks.", metadata={"topic": "deep learning"}),
    Document(page_content="Batch normalisation stabilises training by normalising layer inputs, which allows the use of higher learning rates.", metadata={"topic": "deep learning"}),
    Document(page_content="Learning rate schedulers dynamically adjust the learning rate during training to improve convergence and avoid overshooting.", metadata={"topic": "deep learning"}),
    Document(page_content="Arctic sea ice has declined by about 13% per decade since satellite measurements began in 1979.", metadata={"topic": "climate"}),
    Document(page_content="Carbon capture technology removes CO2 from the atmosphere and stores it underground.", metadata={"topic": "climate"}),
    Document(page_content="The permafrost in Siberia contains vast amounts of methane that could be released as it thaws.", metadata={"topic": "climate"}),
    Document(page_content="The Renaissance was a cultural movement in Europe from the 14th to 17th century that revived classical art.", metadata={"topic": "art"}),
    Document(page_content="Impressionism emerged in 19th-century France, focusing on light, colour, and everyday subjects.", metadata={"topic": "art"}),
    Document(page_content="Abstract expressionism prioritises spontaneous, automatic, and subconscious creation.", metadata={"topic": "art"}),
    Document(page_content="Common law systems derive legal principles from judicial precedent rather than written codes.", metadata={"topic": "law"}),
    Document(page_content="The presumption of innocence requires the prosecution to prove guilt beyond reasonable doubt.", metadata={"topic": "law"}),
    Document(page_content="Intellectual property law protects creations of the mind, including patents, trademarks, and copyrights.", metadata={"topic": "law"}),
]

<h2 style="color:yellow"> 3. Here we have displayed the docs in above using a loop with an enumerate so that we get the index. This ensures that the Documents are correctly loaded.</h2>

In [35]:
documents= [
    Document(
        id = str(uuid4()),
        page_content = item.page_content,
        metadata = item.metadata
    )
    for item in docs
]
print(f"Number of documents created: {len(documents)}")
documents[0].id

Number of documents created: 27


'1310149e-b786-4f9b-b492-4f7530068441'

<h2 style="color:yellow;"> 4. Convert the Documents into Embeddings which is a Numerical vector. Persist is required to save the embeddings. </h2>

In [36]:
# Embed documents and store in an in-memory ChromaDB collection
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=persist_directory,
)

print(f"Number of documents in vectorstore before adding: {len(vectorstore.get()['ids'])}")

if len(vectorstore.get()['ids']) == 0:
    document_ids = vectorstore.add_documents(documents)
    print(f"Document IDs added to vectorstore: {document_ids}")
print(f"Number of documents in vectorstore after adding: {len(vectorstore.get()['ids'])}")

Number of documents in vectorstore before adding: 27
Number of documents in vectorstore after adding: 27


## 5. Read the Stored Data Back ##

In [ ]:
## Read the Stored Data Back ##
raw_records = vectorstore.get(include=["metadatas", "documents", "embeddings"])
print(f"Number of records retrieved: {len(raw_records['ids'])}")
print(f"Sample record metadata: {raw_records['metadatas'][0]}")

Number of records retrieved: 27
Sample record metadata: {'topic': 'space'}


## 6. Run a Similarity Search ##

In [44]:
# Similarity search retriever returns the k most semantically similar documents
# It ranks by cosine similarity between the query embedding and document embeddings

retriever = vectorstore.as_retriever(
    search_type="mmr",  # similarity, similarity_score_threshold, mmr
    search_kwargs={"k": 2},   # k is the number of retrieved docs
)

In [45]:
# this method is not used generally inside chains, or if you require customizations then 
# also you do not use this method

query = "How do cells generate their energy?"

# results = vectorstore.similarity_search(query, k=3)
results = retriever.invoke(query)

print(f"{query}\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i} topic={doc.metadata['topic']}")
    print(f"{doc.page_content}")
    print()

How do cells generate their energy?

Result 1 topic=biology
Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.

Result 2 topic=space
Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.



In [43]:
query = "How do rockets work?"

# retrievers in langchain are runnables, create chains using retrievers
results = retriever.invoke(query)

# All returned documents should be from the space topic
print(f"{query}\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i} topic={doc.metadata['topic']}")
    print(f"{doc.page_content}")
    print()

How do rockets work?

Result 1 topic=space
Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.

Result 2 topic=space
Spacecraft use gravitational slingshots around planets to gain speed without burning extra fuel.

